# Reduced-feature physiology–MWL correlations

This notebook repeats the phase × group exploratory Spearman analysis after an outcome-independent redundancy filter. Feature selection uses only feature missingness and feature–feature Spearman correlations calculated once across all groups and phases. MWL, group-specific effects, p-values, and prior findings are not used for selection.

Training blocks include repeated observations from participants. Correlation p-values and FDR results remain exploratory rather than confirmatory repeated-measures inference.

## Imports, configuration, and paths

In [3]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
pd.set_option("display.max_columns",120); pd.set_option("display.width",200)
CORRELATION_METHOD="spearman"; HIGH_CORRELATION_THRESHOLD=.80; FDR_ALPHA=.05
INCLUDE_IMPUTED_MWL=True; SAVE_TABLES=True; SAVE_FIGURES=True; TOP_N=10
TIE_RTOL=1e-12; TIE_ATOL=1e-12
PHASES=["pre_test","test_1","test_2","test_3","evaluation"]; GROUPS=["Haptic","NoHA"]; MODALITIES=["ECG","EDA","RESP","TEMP","fNIRS"]
assert CORRELATION_METHOD=="spearman"
def root(start=None):
 start=Path.cwd() if start is None else Path(start).resolve()
 for p in (start,*start.parents):
  if (p/"outputs/final_features/physiology_mwl_analysis_dataset.csv").exists(): return p
 raise FileNotFoundError("repository root not found")
ROOT=root(); INPUT=ROOT/"outputs/final_features/physiology_mwl_analysis_dataset.csv"; TAB=ROOT/"postprocessing/outputs/tables"; FIG=ROOT/"postprocessing/outputs/figures"; TAB.mkdir(parents=True,exist_ok=True); FIG.mkdir(parents=True,exist_ok=True)
print("Input:",INPUT)

Input: /Users/gabrieleluzzani/github_repos/HelicopterHapticTrainingPhysio/outputs/final_features/physiology_mwl_analysis_dataset.csv


## Initial 79-feature set and constant-feature audit

In [5]:
all_data=pd.read_csv(INPUT)
PREFIX={"ECG":"delta_ecg_","EDA":"delta_eda_","RESP":"delta_resp_","TEMP":"delta_temp_","fNIRS":"fnirs_"}
modality_features={m:[c for c in all_data if c.startswith(p)] for m,p in PREFIX.items()}; original=[f for m in MODALITIES for f in modality_features[m]]; modality={f:m for m,fs in modality_features.items() for f in fs}
assert len(original)==79 and len(set(original))==79
assert not all_data.duplicated(["participant_id","phase","block_index"]).any()
selection_data=all_data[all_data.phase.isin(["test_1","test_2","test_3","evaluation"])].copy()
nonmissing=selection_data[original].notna().sum(); missing_fraction=selection_data[original].isna().mean(); constants=[f for f in original if selection_data[f].dropna().nunique()<2]
expected_constants=["delta_ecg_lf_hf_std","delta_ecg_phf_std","delta_ecg_plf_std"]
assert sorted(constants)==sorted(expected_constants),f"Unexpected constant set: {constants}"
informative=[f for f in original if f not in constants]
assert len(informative)==76
print("Original",len(original),"constant/unusable",len(constants),"informative",len(informative)); print("Constants:",constants)

Original 79 constant/unusable 3 informative 76
Constants: ['delta_ecg_lf_hf_std', 'delta_ecg_phf_std', 'delta_ecg_plf_std']


## Global pairwise feature–feature Spearman matrix

The matrix is calculated once on the complete dataset using pairwise-complete observations.

In [7]:
corr=selection_data[informative].corr(method="spearman",min_periods=3)
assert corr.shape==(76,76) and np.allclose(np.diag(corr),1,equal_nan=False)
if SAVE_TABLES: corr.to_csv(TAB/"physiological_feature_feature_spearman_matrix_080.csv",index_label="feature")
upper=np.triu(np.ones(corr.shape,dtype=bool),1); initial_high=int(((corr.abs()>=HIGH_CORRELATION_THRESHOLD)&upper).sum().sum())
print("Initial high-correlation pairs:",initial_high)

Initial high-correlation pairs: 42


## Deterministic iterative filter

At each step the currently highest absolute-correlation pair is selected; correlation ties use lexicographic pair order. Greater missingness is removed first. With equal missingness, the feature with higher mean absolute correlation to all remaining features is removed. If those values tie numerically, the lexicographically later feature is removed.

In [9]:
remaining=list(informative); removal={}; step=0
while True:
 sub=corr.loc[remaining,remaining].abs(); pairs=[]
 for i,a in enumerate(remaining):
  for b in remaining[i+1:]:
   value=sub.loc[a,b]
   if pd.notna(value) and value>=HIGH_CORRELATION_THRESHOLD: pairs.append((float(value),min(a,b),max(a,b)))
 if not pairs: break
 pair_rho,a,b=sorted(pairs,key=lambda x:(-x[0],x[1],x[2]))[0]
 ma=int(selection_data[a].isna().sum()); mb=int(selection_data[b].isna().sum())
 mean_a=float(sub.loc[a,[x for x in remaining if x!=a]].mean()); mean_b=float(sub.loc[b,[x for x in remaining if x!=b]].mean())
 if ma!=mb: removed,representative=(a,b) if ma>mb else (b,a)
 elif not np.isclose(mean_a,mean_b,rtol=TIE_RTOL,atol=TIE_ATOL): removed,representative=(a,b) if mean_a>mean_b else (b,a)
 else: removed,representative=max(a,b),min(a,b)
 step+=1; removal[removed]={"removal_step":step,"removed_because_correlated_with":representative,"pair_abs_rho_at_removal":pair_rho,"mean_abs_rho_at_removal":mean_a if removed==a else mean_b,"same_modality_as_representative":modality[removed]==modality[representative]}; remaining.remove(removed)
retained=remaining
retained_corr_values=corr.loc[retained,retained].abs().to_numpy(copy=True); np.fill_diagonal(retained_corr_values,np.nan); maximum_remaining=float(np.nanmax(retained_corr_values))
assert maximum_remaining < HIGH_CORRELATION_THRESHOLD
rows=[]
for f in original:
 status="excluded_constant" if f in constants else ("excluded_high_correlation" if f in removal else "retained"); detail=removal.get(f,{})
 rows.append({"feature":f,"modality":modality[f],"n_nonmissing":int(nonmissing[f]),"missing_fraction":missing_fraction[f],"selection_status":status,"retained":f in retained,"removal_step":detail.get("removal_step",np.nan),"removed_because_correlated_with":detail.get("removed_because_correlated_with",""),"pair_abs_rho_at_removal":detail.get("pair_abs_rho_at_removal",np.nan),"mean_abs_rho_at_removal":detail.get("mean_abs_rho_at_removal",np.nan),"same_modality_as_representative":detail.get("same_modality_as_representative",np.nan)})
audit=pd.DataFrame(rows); reduced=pd.DataFrame({"feature":retained,"modality":[modality[f] for f in retained]})
notebook05_reduced=pd.read_csv(TAB/"physiological_features_reduced_set_no_pretest_080.csv")
retained_matches_notebook05=retained==notebook05_reduced.feature.tolist()
assert retained_matches_notebook05, "The phase analysis retained set differs from notebook 05"
if SAVE_TABLES:
 audit.to_csv(TAB/"physiological_feature_correlation_filter_audit_080.csv",index=False); reduced.to_csv(TAB/"physiological_features_reduced_set_080.csv",index=False)
print("Original=79; constants excluded=",len(constants),"; informative=",len(informative),"; high-correlation exclusions=",len(removal),"; retained=",len(retained)); print("Retained by modality:",reduced.groupby("modality").size().reindex(MODALITIES,fill_value=0).to_dict()); print("Maximum retained pairwise |rho|:",maximum_remaining); print("Cross-modality removals:",sum(not x["same_modality_as_representative"] for x in removal.values()))

Original=79; constants excluded= 3 ; informative= 76 ; high-correlation exclusions= 25 ; retained= 51
Retained by modality: {'ECG': 4, 'EDA': 6, 'RESP': 11, 'TEMP': 5, 'fNIRS': 25}
Maximum retained pairwise |rho|: 0.7762969411979085
Cross-modality removals: 0


## Reduced phase × group physiology–MWL correlations and ten separate FDR families

In [11]:
data=all_data.copy() if INCLUDE_IMPUTED_MWL else all_data[all_data.mwl_source!="imputed_previous"].copy()
def bh(p):
 p=np.asarray(p,float); out=np.full(p.shape,np.nan); mask=np.isfinite(p)
 if not mask.any(): return out
 v=p[mask]; order=np.argsort(v); ranked=v[order]; n=len(v); q=np.minimum.accumulate((ranked*n/np.arange(1,n+1))[::-1])[::-1]; restored=np.empty(n); restored[order]=np.clip(q,0,1); out[np.flatnonzero(mask)]=restored; return out
def one(frame,phase,group,f):
 d=frame[["participant_id","mwl_value",f]].dropna(); base=dict(phase=phase,group=group,feature=f,modality=modality[f],n_observations=len(d),n_participants=d.participant_id.nunique(),n_unique_mwl=d.mwl_value.nunique())
 if len(d)<3 or d.participant_id.nunique()<2:return {**base,"rho":np.nan,"p_value":np.nan,"status":"insufficient_data"}
 if d.mwl_value.nunique()<2:return {**base,"rho":np.nan,"p_value":np.nan,"status":"constant_mwl"}
 if d[f].nunique()<2:return {**base,"rho":np.nan,"p_value":np.nan,"status":"constant_feature"}
 r=spearmanr(d[f],d.mwl_value,nan_policy="omit");return {**base,"rho":float(r.statistic),"p_value":float(r.pvalue),"status":"ok"}
def calculate(frame):
 out=[]
 for phase in PHASES:
  for group in GROUPS:
   fam=frame[(frame.phase==phase)&(frame.group==group)]; r=pd.DataFrame([one(fam,phase,group,f) for f in retained]);r["p_fdr"]=bh(r.p_value);r["significant_nominal"]=r.p_value.lt(FDR_ALPHA);r["significant_fdr"]=r.p_fdr.lt(FDR_ALPHA);out.append(r)
 return pd.concat(out,ignore_index=True)
results=calculate(data);results=results[["phase","group","feature","modality","n_observations","n_participants","n_unique_mwl","rho","p_value","p_fdr","significant_nominal","significant_fdr","status"]]
assert len(results)==10*len(retained)
if SAVE_TABLES:results.to_csv(TAB/"physiology_mwl_spearman_by_phase_group_reduced_080.csv",index=False)
print(results.groupby(["phase","group","status"]).size().to_string())

phase       group   status
evaluation  Haptic  ok        51
            NoHA    ok        51
pre_test    Haptic  ok        51
            NoHA    ok        51
test_1      Haptic  ok        51
            NoHA    ok        51
test_2      Haptic  ok        51
            NoHA    ok        51
test_3      Haptic  ok        51
            NoHA    ok        51


## Reduced-analysis summary and descriptive group differences

In [13]:
summary=[]
for phase in PHASES:
 for group in GROUPS:
  fam=results[(results.phase==phase)&(results.group==group)];src=data[(data.phase==phase)&(data.group==group)];v=fam[fam.status=="ok"];pos=v.rho.idxmax();neg=v.rho.idxmin();ab=v.rho.abs().idxmax()
  summary.append(dict(phase=phase,group=group,observations=len(src),participants=src.participant_id.nunique(),retained_features=len(retained),valid_correlations=len(v),nominal_p_lt_0_05=int(fam.significant_nominal.sum()),fdr_q_lt_0_05=int(fam.significant_fdr.sum()),strongest_positive_feature=v.loc[pos,"feature"],strongest_positive_rho=v.loc[pos,"rho"],strongest_negative_feature=v.loc[neg,"feature"],strongest_negative_rho=v.loc[neg,"rho"],strongest_absolute_feature=v.loc[ab,"feature"],strongest_absolute_rho=v.loc[ab,"rho"],fdr_modalities=";".join(sorted(fam.loc[fam.significant_fdr,"modality"].unique()))))
summary=pd.DataFrame(summary)
h=results[results.group=="Haptic"][["phase","feature","modality","rho","status"]].rename(columns={"rho":"rho_haptic","status":"status_haptic"});n=results[results.group=="NoHA"][["phase","feature","rho","status"]].rename(columns={"rho":"rho_noha","status":"status_noha"});diff=h.merge(n,on=["phase","feature"],validate="one_to_one");diff["delta_rho"]=diff.rho_haptic-diff.rho_noha;diff["abs_delta_rho"]=diff.delta_rho.abs();diff["comparison_status"]=np.where(diff[["rho_haptic","rho_noha"]].notna().all(axis=1),"descriptive_available","undefined_source_correlation")
if SAVE_TABLES:
 summary.to_csv(TAB/"physiology_mwl_correlation_phase_summary_reduced_080.csv",index=False);diff.to_csv(TAB/"physiology_mwl_haptic_noha_rho_differences_by_phase_reduced_080.csv",index=False)
print(summary.to_string(index=False))
for phase in PHASES:print("\n",phase,"largest descriptive differences\n",diff[diff.phase==phase].nlargest(TOP_N,"abs_delta_rho")[["feature","modality","rho_haptic","rho_noha","delta_rho"]].to_string(index=False))

     phase  group  observations  participants  retained_features  valid_correlations  nominal_p_lt_0_05  fdr_q_lt_0_05   strongest_positive_feature  strongest_positive_rho strongest_negative_feature  strongest_negative_rho   strongest_absolute_feature  strongest_absolute_rho      fdr_modalities
  pre_test Haptic            11            11                 51                  51                  0              0       fnirs_deoxy_power_band                0.593926            delta_ecg_pnn50               -0.595465              delta_ecg_pnn50               -0.595465                    
  pre_test   NoHA            10            10                 51                  51                  1              1          delta_eda_scl_slope                0.604599          delta_eda_scl_std               -0.891316            delta_eda_scl_std               -0.891316                 EDA
    test_1 Haptic            22            11                 51                  51                  0         

## Reduced-feature heatmaps

In [15]:
ordered=[f for m in MODALITIES for f in retained if modality[f]==m];bounds=np.cumsum([sum(modality[f]==m for f in ordered) for m in MODALITIES])[:-1];starts=np.r_[0,bounds];ends=np.r_[bounds,len(ordered)]
fig,axes=plt.subplots(1,2,figsize=(14,max(10,len(ordered)*.25)),sharey=True)
for ax,group in zip(axes,GROUPS):
 s=results[results.group==group];mat=s.pivot(index="feature",columns="phase",values="rho").reindex(index=ordered,columns=PHASES);sig=s.pivot(index="feature",columns="phase",values="significant_fdr").reindex(index=ordered,columns=PHASES);im=ax.imshow(mat.to_numpy(float),aspect="auto",cmap="RdBu_r",vmin=-1,vmax=1);ax.set_xticks(range(5),PHASES,rotation=25,ha="right");ax.set_title(group)
 for i in range(len(ordered)):
  for j in range(5):
   if bool(sig.iloc[i,j]):ax.text(j,i,"★",ha="center",va="center",fontsize=8)
 for b in bounds:ax.axhline(b-.5,color="black",lw=1.2)
axes[0].set_yticks(range(len(ordered)),ordered,fontsize=6)
for m,a,b in zip(MODALITIES,starts,ends):axes[0].text(-.53,(a+b-1)/2,m,transform=axes[0].get_yaxis_transform(),ha="right",va="center",fontweight="bold",fontsize=8)
fig.suptitle("Reduced-feature physiology–MWL Spearman rho\n★ family-specific FDR q < 0.05",y=.995);fig.colorbar(im,ax=axes,fraction=.02,pad=.025,label="Spearman rho");fig.subplots_adjust(left=.34,right=.92,top=.95,bottom=.06,wspace=.08)
if SAVE_FIGURES:fig.savefig(FIG/"physiology_mwl_correlations_by_phase_group_reduced_080.png",dpi=200,bbox_inches="tight")
plt.show()
mat=diff.pivot(index="feature",columns="phase",values="delta_rho").reindex(index=ordered,columns=PHASES);fig,ax=plt.subplots(figsize=(9,max(10,len(ordered)*.25)));im=ax.imshow(mat.to_numpy(float),aspect="auto",cmap="RdBu_r",vmin=-1,vmax=1);ax.set_xticks(range(5),PHASES,rotation=25,ha="right");ax.set_yticks(range(len(ordered)),ordered,fontsize=6);ax.set_title("Descriptive Haptic–NoHA difference in Spearman rho\n(no formal difference test)")
for b in bounds:ax.axhline(b-.5,color="black",lw=1.2)
for m,a,b in zip(MODALITIES,starts,ends):ax.text(-.53,(a+b-1)/2,m,transform=ax.get_yaxis_transform(),ha="right",va="center",fontweight="bold",fontsize=8)
fig.colorbar(im,ax=ax,fraction=.025,pad=.03,label="rho_Haptic − rho_NoHA");fig.subplots_adjust(left=.45,right=.91,top=.94,bottom=.06)
if SAVE_FIGURES:fig.savefig(FIG/"physiology_mwl_haptic_noha_rho_difference_by_phase_reduced_080.png",dpi=200,bbox_inches="tight")
plt.show()

04_physiology_mwl_correlations_reduced_features.ipynb:cell14:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
04_physiology_mwl_correlations_reduced_features.ipynb:cell14:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## Full-versus-reduced transparency comparison

In [17]:
full=pd.read_csv(TAB/"physiology_mwl_spearman_by_phase_group.csv");comparison=[]
for phase in PHASES:
 for group in GROUPS:
  f=full[(full.phase==phase)&(full.group==group)];r=results[(results.phase==phase)&(results.group==group)];rfdr=r[r.significant_fdr].feature.tolist();lookup=f.set_index("feature")
  comparison.append(dict(phase=phase,group=group,valid_tests_full=int((f.status=="ok").sum()),valid_tests_reduced=int((r.status=="ok").sum()),nominal_full=int(f.significant_nominal.sum()),nominal_reduced=int(r.significant_nominal.sum()),fdr_full=int(f.significant_fdr.sum()),fdr_reduced=int(r.significant_fdr.sum()),reduced_fdr_features=";".join(rfdr),reduced_fdr_already_nominal_full=";".join(x for x in rfdr if bool(lookup.loc[x,"significant_nominal"])),reduced_fdr_already_fdr_full=";".join(x for x in rfdr if bool(lookup.loc[x,"significant_fdr"]))))
comparison=pd.DataFrame(comparison)
if SAVE_TABLES:comparison.to_csv(TAB/"physiology_mwl_full_vs_reduced_feature_analysis_summary_080.csv",index=False)
print(comparison.to_string(index=False));print("Changes in significance after unsupervised filtering are transparency results, not evidence that filtering improved the analysis.")

     phase  group  valid_tests_full  valid_tests_reduced  nominal_full  nominal_reduced  fdr_full  fdr_reduced                                                                                                                                                                                                        reduced_fdr_features                                                                                                                                                                                            reduced_fdr_already_nominal_full                                                                                                                                                                reduced_fdr_already_fdr_full
  pre_test Haptic                76                   51             2                0         0            0                                                                                                                                                       

## Approved-imputation sensitivity and final report

In [19]:
a=calculate(all_data)[["phase","group","feature","rho"]].rename(columns={"rho":"rho_with_imputation"});b=calculate(all_data[all_data.mwl_source!="imputed_previous"])[["phase","group","feature","rho"]].rename(columns={"rho":"rho_observed_only"});sens=a.merge(b,on=["phase","group","feature"],validate="one_to_one");sens["absolute_change_rho"]=(sens.rho_with_imputation-sens.rho_observed_only).abs()
if SAVE_TABLES:sens.to_csv(TAB/"physiology_mwl_phase_group_imputation_sensitivity_reduced_080.csv",index=False)
print("Imputation sensitivity max",sens.absolute_change_rho.max(),"median",sens.absolute_change_rho.median())
print("\nFINAL REDUCED ANALYSIS SUMMARY");print(summary[["phase","group","observations","participants","retained_features","valid_correlations","nominal_p_lt_0_05","fdr_q_lt_0_05","strongest_absolute_feature","strongest_absolute_rho","fdr_modalities"]].to_string(index=False));print("Repeated blocks remain repeated observations; no group or temporal difference in rho was formally tested.")

Imputation sensitivity max 0.1627932624346038 median 0.0

FINAL REDUCED ANALYSIS SUMMARY
     phase  group  observations  participants  retained_features  valid_correlations  nominal_p_lt_0_05  fdr_q_lt_0_05   strongest_absolute_feature  strongest_absolute_rho      fdr_modalities
  pre_test Haptic            11            11                 51                  51                  0              0              delta_ecg_pnn50               -0.595465                    
  pre_test   NoHA            10            10                 51                  51                  1              1            delta_eda_scl_std               -0.891316                 EDA
    test_1 Haptic            22            11                 51                  51                  0              0              delta_temp_mean               -0.416556                    
    test_1   NoHA            20            10                 51                  51                  3              0    delta_resp_amplitude_